# 08m — Teste do Random Forest multiclasse AJUSTADO (modelo final da busca 07m), uma única vez

Este notebook avalia no **conjunto de teste** o **modelo final (refit) escolhido pela busca de hiperparâmetros** (notebook 07m: 100 árvores, profundidade livre, `min_samples_leaf = 5`, `class_weight="balanced"`) e **compara com o modelo padrão** já avaliado no notebook 06m, **nas mesmas linhas**. Nada é treinado nem ajustado aqui, e o teste é usado **uma única vez** para este modelo.

## O que é avaliado (igual ao 06m)
| Item | Como |
|---|---|
| **Dados** | `prep_vw_rf_multi_teste_sem_sobrep`: as 7.216.487 linhas do outro dia, sem sobreposição com o treino e só das 7 classes. Mistura natural (o `Syn` é ~68%), lidas em lotes |
| **Previsão** | A classe de maior probabilidade |
| **Métricas** | Recall, precisão e F1 por classe, acurácia balanceada, F1 macro (também sem o `UDPLag`), MCC, ROC-AUC um-contra-resto, matriz de confusão 7×7, validação × teste |
| **`Portmap`** | Classe que só existe no teste: fica **fora das métricas principais** e tem uma seção própria (para qual classe é empurrada e com que confiança) |

## O que este notebook acrescenta: a comparação com o 06m (seção 6)
Compara o modelo ajustado com o padrão **no mesmo teste**: métricas gerais, recall, precisão e F1 por classe, o desempenho **ataque × BENIGN** (juntando as 6 classes de ataque, um cálculo posterior) e uma comparação **linha a linha** (quantas linhas só um dos dois modelos acerta, e para onde mudaram as previsões do `Syn`). É **descritiva**: o teste tem uma só partição e muitas linhas repetidas.

## Garantias (verificadas por código, não só prometidas)
1. **Nenhum modelo é treinado.** O `RandomForestClassifier.fit` é substituído por uma função que **dá erro** se for chamada. O modelo é só **carregado** do disco.
2. **Nenhum arquivo existente é sobrescrito, alterado ou apagado.** Gravar CSV, JSON, NPZ, PNG ou modelo, substituir ou apagar um arquivo que já existe **dentro da pasta do modelo** dá erro. Cada arquivo só é **criado se ainda não existir**, e o que já existia é mantido.
3. **O servidor só é lido.** Este notebook só faz `SELECT`, e a trava de escrita do início só permite tabelas `prep_*`.
4. **Conferência final:** o notebook compara o tamanho e a data de **todos os arquivos que já existiam** na pasta (os 27 da busca, os modelos, os CSVs do 07m) antes e depois, e só passa se **nenhum mudou**.
5. **Uso único do teste:** o `metricas_teste.json` é gravado **por último**, depois das conferências. Se existir, o notebook **para**.

## O que é criado (só arquivos novos, na pasta `rfm_busca_<data>` do modelo)
`predicoes_teste.npz`, `teste_lotes/` (probabilidades de cada lote; base do ponto de retomada), `metricas_teste.csv/json`, `metricas_por_classe_teste.csv`, `matriz_confusao_teste.csv`, `validacao_vs_teste.csv`, `portmap_teste.csv`, `custo_inferencia_teste.json`, `tempos_teste.csv`, `comparacao_ajustado_vs_padrao_*.csv`, o gráfico e uma linha em `registro_testes.csv`.

## Antes de rodar
- O 07m precisa ter terminado (a pasta da busca precisa ter o arquivo `FINALIZADO`) e o 06m precisa ter sido rodado (para a comparação).
- O `.env` precisa apontar para o servidor novo.
- Coloque este arquivo na **mesma pasta** dos outros (onde está o `.env`). Tempo: parecido com o do 06m (~5 min).
- Se a busca finalizada não for a mais recente, defina `PASTA_RUN = r"...\rfm_busca_<data>"` antes da célula 1.1.


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas do teste
- **`PreprocessadorRF`**: o pré-processamento do RF (igual ao da view), usado para as classes que o modelo não conhece.
- **`ler_csv_com_espera`**, **`Barra`**, **`fmt_tempo`**: leitura do teste em CSV (com espera se o servidor ficar sem memória) e barra de progresso.
- **`metricas_multi_grande`**: métricas multiclasse para arrays grandes (confusão por contagem), e o intervalo de Wilson.

In [5]:
class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

# ---------- métricas multiclasse para arrays grandes (sem cópias em float64; confusão por contagem) ----------
def metricas_multi_grande(y, P):
    y = np.asarray(y, dtype=np.int64); K_ = P.shape[1]; pred = P.argmax(axis=1)
    cm = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(np.float64)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, np.ascontiguousarray(P[:, k], dtype=np.float64))
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t, "previstas": p}

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (c - h, c + h)

# autoteste (não toca no servidor): confere contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc
_r = np.random.default_rng(1); _y = _r.integers(0, 4, 500); _P = _r.dirichlet(np.ones(4), 500); _P[np.arange(500), _y] += 0.3; _P = (_P / _P.sum(axis=1, keepdims=True)).astype(np.float32); _pr = _P.argmax(axis=1)
_e, _a = metricas_multi_grande(_y, _P)
assert abs(_e["acuracia"] - _acc(_y, _pr)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr)) < 1e-12 and abs(_e["f1_macro"] - _f1(_y, _pr, average="macro")) < 1e-12 and abs(_e["mcc"] - _mcc(_y, _pr)) < 1e-12, _e
assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P.astype(np.float64), multi_class="ovr", average="macro")) < 1e-9
_lo, _hi = wilson(50, 100); assert 0.40 < _lo < 0.41 and 0.59 < _hi < 0.60
print("Autoteste das ferramentas do teste multiclasse OK")

# ---------- proteções: o que este notebook NUNCA faz (treinar, sobrescrever, apagar) ----------
import warnings, shutil
def silenciar_aviso_sklearn():
    # Com muitos núcleos, o scikit-learn às vezes deixa a lista de filtros de aviso VAZIA (condição de corrida entre threads) e passa a emitir, a cada árvore, um aviso inofensivo
    # sobre `delayed`/`Parallel`. Este desvio descarta SÓ essa mensagem; os outros avisos continuam funcionando. Não muda nenhum resultado.
    warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
    if not getattr(warnings.warn, "_filtrado_sklearn", False):
        _orig = warnings.warn
        def _warn_filtrado(message, *args, **kwargs):
            if isinstance(message, str) and message.startswith("`sklearn.utils.parallel.delayed` should be used with"): return
            return _orig(message, *args, **kwargs)
        _warn_filtrado._filtrado_sklearn = True; warnings.warn = _warn_filtrado
silenciar_aviso_sklearn()

TREINOS_BLOQUEADOS = []
CRIADOS, MANTIDOS = [], []
def instalar_travas(pasta, permitir_regravar=("tempos_teste.csv",)):
    # (1) bloqueia o treino do RF; (2) bloqueia qualquer gravação que SOBRESCREVA um arquivo existente dentro de `pasta`, e qualquer remoção de arquivo dentro dela.
    # Só `tempos_teste.csv` (o registro de tempo, que este notebook cria e atualiza a cada lote) e arquivos temporários (.tmp) podem ser regravados.
    from sklearn.ensemble import RandomForestClassifier as _RF
    import numpy as _np, pandas as _pd, joblib as _jl, pathlib as _pl
    raiz = str(Path(pasta).resolve()); perm = set(permitir_regravar)
    def dentro(alvo):
        try: return str(Path(alvo).resolve()).startswith(raiz)
        except Exception: return False
    def checa(alvo, acao, permitir_acrescimo=False):
        if not isinstance(alvo, (str, os.PathLike)) or not dentro(alvo): return
        p = Path(alvo).resolve()
        if p.exists() and p.name not in perm and ".tmp" not in p.name and not permitir_acrescimo:
            raise PermissionError(f"BLOQUEADO: {acao} sobrescreveria '{p.name}', que já existe na pasta do modelo. Este notebook só cria arquivos novos.")
    def embrulha(obj, nome, pos, acao, kw="", acrescimo_ok=False):
        orig = getattr(obj, nome)
        if getattr(orig, "_travado", False): return
        def f(*a, **k):
            alvo = a[pos] if len(a) > pos else k.get(kw)
            checa(alvo, acao, permitir_acrescimo=(acrescimo_ok and k.get("mode") == "a")); return orig(*a, **k)
        f._travado = True; f.__name__ = getattr(orig, "__name__", nome); setattr(obj, nome, f)
    embrulha(_pd.DataFrame, "to_csv", 1, "gravar CSV", "path_or_buf", acrescimo_ok=True); embrulha(_np, "savez", 0, "gravar NPZ", "file"); embrulha(_np, "savez_compressed", 0, "gravar NPZ", "file")
    embrulha(_jl, "dump", 1, "gravar modelo", "filename"); embrulha(_pl.Path, "write_text", 0, "gravar arquivo"); embrulha(_pl.Path, "write_bytes", 0, "gravar arquivo")
    embrulha(os, "replace", 1, "substituir arquivo"); embrulha(os, "rename", 1, "renomear arquivo")
    try:
        from matplotlib.figure import Figure; embrulha(Figure, "savefig", 1, "gravar figura", "fname")
    except ImportError: pass
    def _proibe_remocao(orig):
        def f(*a, **k):
            alvo = a[0] if a else None
            if isinstance(alvo, (str, os.PathLike)) and dentro(alvo): raise PermissionError(f"BLOQUEADO: apagar '{Path(alvo).name}' dentro da pasta do modelo. Este notebook nunca apaga arquivos.")
            return orig(*a, **k)
        f._travado = True; return f
    if not getattr(os.remove, "_travado", False): os.remove = _proibe_remocao(os.remove)
    if not getattr(_pl.Path.unlink, "_travado", False):
        _unl = _pl.Path.unlink
        def _u(self, *a, **k):
            if dentro(self): raise PermissionError(f"BLOQUEADO: apagar '{self.name}' dentro da pasta do modelo. Este notebook nunca apaga arquivos.")
            return _unl(self, *a, **k)
        _u._travado = True; _pl.Path.unlink = _u
    if not getattr(_RF.fit, "_bloqueado", False):
        def _fit_bloqueado(self, *a, **k):
            TREINOS_BLOQUEADOS.append(1); raise RuntimeError("BLOQUEADO: treinar um RF não é permitido neste notebook (ele só carrega o modelo já treinado).")
        _fit_bloqueado._bloqueado = True; _RF.fit = _fit_bloqueado

def gravar(nome, escrever):
    # Cria o arquivo SÓ se ele ainda NÃO existir (nunca sobrescreve). Registra o que foi criado e o que foi mantido.
    caminho = PASTA_RUN / nome
    if caminho.exists(): MANTIDOS.append(nome); return False
    escrever(caminho); CRIADOS.append(nome); return True
def gravar_csv(df, nome, **kw): return gravar(nome, lambda p: df.to_csv(p, **kw))
def gravar_json(obj, nome): return gravar(nome, lambda p: salvar_json(obj, p))


Autoteste das ferramentas do teste OK
Autoteste das ferramentas do teste multiclasse OK


## 1. Parâmetros e carga
Carrega o modelo final do 05m (a pasta `rfm_*` mais recente), confere que o pré-processamento, as colunas, os hiperparâmetros e as classes batem com o que foi gravado, e **que a trava de uso único não foi acionada**. Mostra quantas linhas de cada classe o teste tem e quais classes do teste **o modelo não conhece**.

In [6]:
# ---------- 1.1 Parâmetros e carga ----------
import joblib
from sklearn.metrics import roc_auc_score
PASTA_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_multiclasse_busca"
PASTA_RUN = globals().get("PASTA_RUN", None)
if PASTA_RUN is None:
    runs = sorted(p for p in PASTA_BASE.glob("rfm_busca_*") if (p / "modelo_final_refit.joblib").exists() and (p / "FINALIZADO").exists())
    assert runs, "nenhuma pasta rfm_busca_* FINALIZADA com modelo_final_refit.joblib em " + str(PASTA_BASE) + ": rode antes o notebook 07m (ou defina PASTA_RUN)"
    if len(runs) > 1: print("⚠ há mais de uma busca finalizada; usando a mais recente. Para escolher outra, defina PASTA_RUN:", [p.name for p in runs])
    PASTA_RUN = runs[-1]
PASTA_RUN = Path(PASTA_RUN); assert (PASTA_RUN / "FINALIZADO").exists() and (PASTA_RUN / "modelo_final_refit.joblib").exists(), f"{PASTA_RUN.name} não é uma busca FINALIZADA com modelo final: rode o 07m até o fim"
PASTA_PADRAO = globals().get("PASTA_PADRAO", None)                       # resultado do 06m (modelo padrão), para a comparação da seção 6
if PASTA_PADRAO is None:
    _c = sorted(p for p in (PASTA_BASE.parent / "rf_multiclasse").glob("rfm_*") if (p / "metricas_teste.json").exists() and (p / "predicoes_teste.npz").exists()); PASTA_PADRAO = _c[-1] if _c else None
PASTA_PADRAO = Path(PASTA_PADRAO) if PASTA_PADRAO else None
VIEW_TESTE, TAB_TESTE, TAB_TREINO, TAB_CLASSES = "prep_vw_rf_multi_teste_sem_sobrep", "prep_final_teste", "prep_final_treino", "prep_classes_rf_multi"
LOTE_LINHAS = globals().get("LOTE_LINHAS", 50_000)          # ~50 mil linhas por lote: pouca memória por vez
PERMITIR_REPETIR = globals().get("PERMITIR_REPETIR", False)
SEMENTE = globals().get("SEMENTE", 42)
T_SEC = {}

RESULTADO = PASTA_RUN / "metricas_teste.json"
if RESULTADO.exists() and not PERMITIR_REPETIR:
    raise RuntimeError(f"O teste já foi avaliado para esta execução ({RESULTADO.name}). Repetir o teste depois de ajustar o modelo o invalida como avaliação independente. "
                       "Para repetir de propósito (por exemplo, para refazer os gráficos), defina PERMITIR_REPETIR = True antes desta célula.")
assert existe(VIEW_TESTE) and existe(TAB_CLASSES), f"faltam {VIEW_TESTE} ou {TAB_CLASSES}: rode o notebook 03m"
# fotografia dos arquivos que JÁ EXISTEM (tamanho e data) e proteções: sem treino, sem sobrescrever, sem apagar
SNAP_ANTES = {str(f.relative_to(PASTA_RUN)): (f.stat().st_size, f.stat().st_mtime_ns) for f in PASTA_RUN.rglob("*") if f.is_file()}
instalar_travas(PASTA_RUN); silenciar_aviso_sklearn()
print(f"🔒 Proteções ativas: treino de RF BLOQUEADO; sobrescrita e remoção de arquivos BLOQUEADAS em {PASTA_RUN.name} ({len(SNAP_ANTES)} arquivos já existem e serão conferidos no final); o servidor só é lido.")

modelo = joblib.load(PASTA_RUN / "modelo_final_refit.joblib"); pc = json.loads((PASTA_RUN / "preprocessamento.json").read_text(encoding="utf-8"))
config = json.loads((PASTA_RUN / "config_treino.json").read_text(encoding="utf-8")); cls_gravado = json.loads((PASTA_RUN / "classes.json").read_text(encoding="utf-8"))
prep = PreprocessadorRF(pc["features"], pc["medianas_treino"], pc["colunas_retiradas_ablacao"], pc["versao_views_rf"]); CM = prep.colunas_modelo
assert prep.colunas_modelo == pc["colunas_modelo"] == list(modelo.colunas_modelo_), "a ordem das colunas do modelo difere da gravada"
assert all(getattr(modelo, k) == v for k, v in config["hiperparametros"].items()), "o modelo carregado não tem os hiperparâmetros gravados"
CLASSES = cls_gravado["classes"]; COD = cls_gravado["codigos"]; K = len(CLASSES); assert list(modelo.classes_) == list(range(K)) and [COD[c] for c in CLASSES] == list(range(K))
cols_view = colunas_view(VIEW_TESTE); assert cols_view[:2] == ["id_linha", "y"] and cols_view[2:] == CM, "as colunas da view de teste não coincidem com as do modelo (nome ou ordem)"
cl = consultar_num(f"SELECT codigo, classe, linhas_teste_sem_sobrep FROM {TAB_CLASSES} ORDER BY codigo"); assert cl.classe.tolist() == CLASSES, "as classes da tabela do notebook 03m diferem das do modelo"
N_CLASSE = {c: int(n) for c, n in zip(cl.classe, cl.linhas_teste_sem_sobrep)}; N_TESTE = n_linhas(VIEW_TESTE); assert sum(N_CLASSE.values()) == N_TESTE

nome_tipo = lambda t: re.sub(r"^DrDoS_", "", str(t))
tt = consultar_num(f"SELECT label_multi AS tipo, count() AS n FROM {TAB_TESTE} WHERE sobrepoe_treino = 0 GROUP BY label_multi"); tt["tipo"] = tt["tipo"].map(nome_tipo); tt = tt.groupby("tipo", as_index=False)["n"].sum()
FORA = {r.tipo: int(r.n) for r in tt.itertuples() if r.tipo not in COD}                      # classes do teste que o modelo não conhece (ex.: Portmap)
print(f"Modelo avaliado: {PASTA_RUN.name} | treinado em {config['data']} | {config['hiperparametros']['n_estimators']} árvores, class_weight = {config['hiperparametros']['class_weight']} | {len(CM)} atributos")
print(f"Hiperparâmetros: {config['hiperparametros']}")
print(f"Modelo padrão para comparação (06m): {PASTA_PADRAO.name if PASTA_PADRAO else 'NÃO ENCONTRADO (a seção 6 será pulada)'}")
print(f"Teste (sem sobreposição com o treino, só das {K} classes do modelo): {N_TESTE:,} linhas")
print("  " + " | ".join(f"{c}: {n:,} ({100 * n / N_TESTE:.3f}%)" for c, n in N_CLASSE.items()))
print("Classes do teste que o modelo NÃO conhece (fora das métricas principais; análise à parte):", FORA or "nenhuma")

🔒 Proteções ativas: treino de RF BLOQUEADO; sobrescrita e remoção de arquivos BLOQUEADAS em rfm_busca_20261102_001711 (60 arquivos já existem e serão conferidos no final); o servidor só é lido.
Modelo avaliado: rfm_busca_20261102_001711 | treinado em 2026-11-03T17:36:50 | 100 árvores, class_weight = balanced | 44 atributos
Hiperparâmetros: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
Modelo padrão para comparação (06m): rfm_20261101_210334
Teste (sem sobreposição com o treino, só das 7 classes do modelo): 7,216,487 linhas
  BENIGN: 56,216 (0.779%) | LDAP: 112,491 (1.559%) | MSSQL: 854,242 (11.837%) | NetBIOS: 91,175 (1.263%) | Syn: 4,887,060 (67.721%) | UDP: 1,214,746 (16.833%) | UDPLag: 557 (0.008%)
Classes do teste que o modelo NÃO conhece (fora das métricas principais; análise à parte): {'Portmap': 2702}


## 2. Previsão em lotes
O teste é lido em **lotes de `LOTE_LINHAS` linhas** (faixas do `id_linha`, para o servidor ler pouca coisa de cada vez) e cada lote vira um arquivo em `teste_lotes/` com o id das linhas, a classe real e as **probabilidades das 7 classes**. Se o notebook for interrompido, os lotes prontos são reaproveitados. O tempo de relógio de cada sessão (leitura do servidor + previsão + gravação dos lotes) fica em `tempos_teste.csv`.

In [7]:
# ---------- 2.1 Previsão em lotes ----------
B = max(1, math.ceil(N_TESTE / LOTE_LINHAS)); PASTA_LOTES = PASTA_RUN / "teste_lotes"; PASTA_LOTES.mkdir(exist_ok=True)
cols_sql = ", ".join("v." + q(c) for c in CM); ids_l, ys, Ps, t_pred, n_lote = [], [], [], [], []; reaproveitados = 0
barra = Barra(B, "Previsão no teste"); t_inicio = time.perf_counter()
ARQ_SESSOES = PASTA_RUN / "tempos_teste.csv"      # uma linha por sessão (leitura do servidor + previsão + gravação dos lotes); sobrevive a interrupções
sess_ant = pd.read_csv(ARQ_SESSOES) if ARQ_SESSOES.exists() else pd.DataFrame(columns=["inicio", "fim", "relogio_s", "lotes_novos"])
sess_inicio = datetime.now(); lotes_novos = 0
def gravar_sessao():
    nova = pd.DataFrame([{"inicio": sess_inicio.isoformat(timespec="seconds"), "fim": datetime.now().isoformat(timespec="seconds"), "relogio_s": round(time.perf_counter() - t_inicio, 1), "lotes_novos": lotes_novos}])
    (pd.concat([sess_ant, nova], ignore_index=True) if len(sess_ant) else nova).to_csv(ARQ_SESSOES, index=False)       # sem concatenar com tabela vazia (evita o aviso do pandas)
for j in range(B):
    arq = PASTA_LOTES / f"lote_{j:03d}_de_{B:03d}.npz"
    if arq.exists():
        d = np.load(arq); ids_l.append(d["id"]); ys.append(d["y"]); Ps.append(d["proba"]); t_pred.append(float(d["t"])); n_lote.append(len(d["y"])); reaproveitados += 1
        barra.avancar(1, f"lote {j + 1}/{B}: já pronto"); continue
    lo, hi = j * 2 ** 64 // B, (j + 1) * 2 ** 64 // B
    onde_v = f"v.id_linha >= {lo}" + (f" AND v.id_linha < {hi}" if j < B - 1 else "")
    barra.avancar(0, f"lote {j + 1}/{B}: lendo... (memória do processo {rss_mib():,.0f} MiB)")
    df = ler_csv_com_espera(f"SELECT toString(v.id_linha) AS id, v.y, {cols_sql} FROM {VIEW_TESTE} AS v WHERE {onde_v}", ["id", "y"] + CM, texto=("id",))
    if len(df) == 0:                                            # faixa sem linhas: registra o lote vazio e segue
        np.savez(arq, id=np.zeros(0, np.uint64), y=np.zeros(0, np.uint8), proba=np.zeros((0, K), np.float32), t=0.0)
        ids_l.append(np.zeros(0, np.uint64)); ys.append(np.zeros(0, np.uint8)); Ps.append(np.zeros((0, K), np.float32)); t_pred.append(0.0); n_lote.append(0); barra.avancar(1, f"lote {j + 1}/{B}: faixa sem linhas"); continue
    X, y = df[CM].to_numpy(float), df["y"].to_numpy(np.uint8); assert np.isfinite(X).all(), "valores não finitos no teste"; assert int(y.max()) < K, "classe do teste fora das classes do modelo"
    ids = df["id"].astype("uint64").to_numpy()
    silenciar_aviso_sklearn(); t0 = time.perf_counter(); P = modelo.predict_proba(X).astype(np.float32); dt = time.perf_counter() - t0
    np.savez_compressed(arq, id=ids, y=y, proba=P, t=dt)
    ids_l.append(ids); ys.append(y); Ps.append(P); t_pred.append(dt); n_lote.append(len(y))
    lotes_novos += 1; gravar_sessao()
    barra.avancar(1, f"lote {j + 1}/{B}: {len(y):,} linhas ({dt:.1f} s de previsão, {len(y) / max(dt, 1e-9):,.0f} fluxos/s)")
    del df, X; gc.collect()
barra.fechar()
if lotes_novos: gravar_sessao()
if not ARQ_SESSOES.exists():                                   # todos os lotes já estavam prontos e não há registro de sessão: usa a soma dos tempos de previsão (um piso do tempo real)
    pd.DataFrame([{"inicio": "", "fim": "", "relogio_s": round(float(sum(t_pred)), 1), "lotes_novos": 0}]).to_csv(ARQ_SESSOES, index=False); print("⚠ sem registro de sessão: o tempo de relógio abaixo é só a soma dos tempos de previsão dos lotes.")
_ses = pd.read_csv(ARQ_SESSOES) if ARQ_SESSOES.exists() else pd.DataFrame({"relogio_s": []})
TEMPO_TESTE_RELOGIO = float(_ses["relogio_s"].sum()); N_SESSOES_TESTE = len(_ses)
print(f"\n⏱ Tempo do teste: {fmt_tempo(TEMPO_TESTE_RELOGIO)} de relógio em {N_SESSOES_TESTE} sessão(ões) (leitura do servidor + previsão + gravação dos lotes) | só a previsão: {fmt_tempo(float(sum(t_pred)))}")
if reaproveitados: print(f"  {reaproveitados} lote(s) já estavam prontos e foram reaproveitados")
id_te, y_te, P_te = np.concatenate(ids_l), np.concatenate(ys), np.concatenate(Ps)               # uint64, uint8 e float32: pouca memória
del ids_l, ys, Ps
assert len(y_te) == N_TESTE, f"as linhas dos lotes ({len(y_te):,}) não fecham com o total da view ({N_TESTE:,})"
print(f"\n✔ {len(y_te):,} previsões em {time.perf_counter() - t_inicio:.0f} s | memória do processo {rss_mib():,.0f} MiB")
print({c: int((y_te == COD[c]).sum()) for c in CLASSES})
T_SEC["2.1_lotes"] = time.perf_counter() - t_inicio

  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.17 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.23 GiB); esperando até 30s e tentando de novo (1/5)...
Previsão no teste: concluído em 4min 54s

⏱ Tempo do teste: 4min 54s de relógio em 1 sessão(ões) (leitura do servidor + previsão + gravação dos lotes) | só a previsão: 1min 09s

✔ 7,216,487 previsões em 295 s | memória do processo 729 MiB
{'BENIGN': 56216, 'LDAP': 112491, 'MSSQL': 854242, 'NetBIOS': 91175, 'Syn': 4887060, 'UDP': 1214746, 'UDPLag': 557}


## 3. Métricas no teste
**Previsão = classe de maior probabilidade.** A tabela mostra as métricas gerais, depois **por classe** (com o intervalo de 95% do recall, que é **otimista**: o teste tem muitas linhas repetidas) e a **matriz de confusão**, em contagem, em fração da classe real (a diagonal é o recall) e em fração da classe **prevista** (de onde vêm as previsões de cada classe, que é o que mostra a precisão).

O **`UDPLag`** é tratado à parte: o teste só tem 557 linhas dele, então bastam alguns erros das classes grandes para derrubar a precisão. A célula mostra **de onde vêm as previsões de `UDPLag`** e o **F1 macro também sem o `UDPLag`** (métrica secundária, definida antes de ver o resultado). Por fim, a comparação **validação × teste**.

In [8]:
# ---------- 3.1 Métricas no teste ----------
t_sec0 = time.perf_counter()
esc, arr = metricas_multi_grande(y_te, P_te); pred_te = P_te.argmax(axis=1).astype(np.uint8); cm = arr["cm"]
ordem = ["roc_auc_ovr_macro", "acuracia_balanceada", "acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc"]
ESPECIAL = globals().get("CLASSE_ESPECIAL", "UDPLag"); i_esp = COD.get(ESPECIAL)
f1_sem_esp = float(np.delete(arr["f1"], i_esp).mean()) if i_esp is not None else float("nan"); rec_sem_esp = float(np.nanmean(np.delete(arr["recall"], i_esp))) if i_esp is not None else float("nan")
tab_te = pd.DataFrame({"teste": {k: esc[k] for k in ordem}}); print(f"== Métricas gerais no teste ({N_TESTE:,} linhas; mistura real das classes) =="); display(tab_te.round(5))
if i_esp is not None: print(f"Métricas secundárias SEM a classe {ESPECIAL} na média (as previsões não mudam): F1 macro = {f1_sem_esp:.5f} | acurácia balanceada = {rec_sem_esp:.5f}")

n = arr["suporte"].astype(int); acertos = np.diag(cm).astype(int); ic = [wilson(int(acertos[i]), int(n[i])) for i in range(K)]
tab_cls_te = pd.DataFrame({"linhas no teste": n, "% do teste": 100 * n / n.sum(), "recall": arr["recall"], "IC95% recall (inf.)": [a for a, _ in ic], "IC95% recall (sup.)": [b for _, b in ic], "precisão": arr["precisao"], "F1": arr["f1"],
                           "ROC-AUC um-contra-resto": arr["auc"], "previstas como a classe": arr["previstas"].astype(int)}, index=CLASSES)
print("\n== Por classe (o IC95% é otimista: o teste tem ~89% de linhas repetidas) =="); display(tab_cls_te.round(5))
cm_te = pd.DataFrame(cm.astype(int), index=[f"real {c}" for c in CLASSES], columns=[f"prev. {c}" for c in CLASSES])
cm_lin = pd.DataFrame(cm / cm.sum(axis=1, keepdims=True), index=cm_te.index, columns=cm_te.columns)
cm_col = pd.DataFrame(np.divide(cm, cm.sum(axis=0, keepdims=True), out=np.zeros_like(cm), where=cm.sum(axis=0, keepdims=True) > 0), index=cm_te.index, columns=cm_te.columns)
print("\n== Matriz de confusão no teste (linhas = classe real; colunas = classe prevista) =="); display(cm_te)
print("== Em fração da classe REAL (a diagonal é o recall) =="); display(cm_lin.round(4))
print("== Em fração da classe PREVISTA (a diagonal é a precisão; mostra de onde vêm as previsões de cada classe) =="); display(cm_col.round(4))
fora = sorted([(CLASSES[i], CLASSES[j], int(cm[i, j]), cm[i, j] / cm[i].sum()) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0], key=lambda x: -x[2])[:10]
print("Confusões mais frequentes (real → previsto): " + " | ".join(f"{a}→{b}: {nn:,} ({100 * f:.2f}% de {a})" for a, b, nn, f in fora))
if i_esp is not None:
    comp = pd.DataFrame({"linhas": cm[:, i_esp].astype(int), "% das previsões de " + ESPECIAL: 100 * cm[:, i_esp] / cm[:, i_esp].sum()}, index=CLASSES)
    print(f"\n== De onde vêm as {int(cm[:, i_esp].sum()):,} previsões de {ESPECIAL} (o teste só tem {int(n[i_esp]):,} linhas reais dessa classe) =="); display(comp.round(3))

# validação × teste (a validação é do mesmo dia do treino; a mistura de classes é outra, então precisão e F1 não são comparáveis em valor absoluto)
mv = json.loads((PASTA_RUN / "metricas_validacao.json").read_text(encoding="utf-8")); pcv = mv["por_classe"]
rec_v = [pcv[c]["recall"] for c in CLASSES]; pre_v = [pcv[c]["precisão (sem pesos)"] for c in CLASSES]; f1_v = [pcv[c]["F1 (sem pesos)"] for c in CLASSES]
vs_cls = pd.DataFrame({"recall (validação)": rec_v, "recall (teste)": arr["recall"], "Δ recall": arr["recall"] - np.array(rec_v), "precisão (validação)": pre_v, "precisão (teste)": arr["precisao"], "F1 (validação)": f1_v, "F1 (teste)": arr["f1"]}, index=CLASSES)
vs_ger = pd.DataFrame({"validação (sem pesos)": {k: mv["sem_pesos"][k] for k in ordem}, "teste": {k: esc[k] for k in ordem}}); vs_ger["Δ"] = vs_ger["teste"] - vs_ger["validação (sem pesos)"]
print("\n== Validação × teste: geral =="); display(vs_ger.round(5)); print("== Validação × teste: por classe =="); display(vs_cls.round(4))
T_SEC["3.1_metricas"] = time.perf_counter() - t_sec0

== Métricas gerais no teste (7,216,487 linhas; mistura real das classes) ==


,teste
roc_auc_ovr_macro,0.92483
acuracia_balanceada,0.67714
acuracia,0.33847
precisao_macro,0.56041
f1_macro,0.41477
f1_ponderado,0.42581
mcc,0.36879


Métricas secundárias SEM a classe UDPLag na média (as previsões não mudam): F1 macro = 0.48386 | acurácia balanceada = 0.73045

== Por classe (o IC95% é otimista: o teste tem ~89% de linhas repetidas) ==


,linhas no teste,% do teste,recall,IC95% recall (inf.),IC95% recall (sup.),precisão,F1,ROC-AUC um-contra-resto,previstas como a classe
BENIGN,56216,0.77899,0.98063,0.97946,0.98174,0.02728,0.05308,0.99077,2020853
LDAP,112491,1.55881,0.99258,0.99206,0.99306,0.75974,0.86069,0.99813,146966
MSSQL,854242,11.83737,0.34106,0.34006,0.34207,0.99200,0.50760,0.99777,293698
NetBIOS,91175,1.26343,0.99872,0.99846,0.99893,0.14399,0.25170,0.99957,632371
Syn,4887060,67.72076,0.16168,0.16135,0.16201,0.99990,0.27835,0.99988,790216
UDP,1214746,16.83293,0.90805,0.90754,0.90857,0.99990,0.95177,0.99978,1103166
UDPLag,557,0.00772,0.35727,0.31858,0.39792,0.00009,0.00018,0.48787,2229217



== Matriz de confusão no teste (linhas = classe real; colunas = classe prevista) ==


,prev. BENIGN,prev. LDAP,prev. MSSQL,prev. NetBIOS,prev. Syn,prev. UDP,prev. UDPLag
real BENIGN,55127,246,1,678,74,0,90
real LDAP,154,111656,183,494,2,0,2
real MSSQL,235,35034,291348,527323,1,52,249
real NetBIOS,102,0,9,91058,1,3,2
real Syn,1965048,18,751,2489,790137,8,2128609
real UDP,109,9,1329,10178,1,1103054,100066
real UDPLag,78,3,77,151,0,49,199


... (6 outputs left out)


== Validação × teste: geral ==


,validação (sem pesos),teste,Δ
roc_auc_ovr_macro,0.99492,0.92483,-0.07010
acuracia_balanceada,0.92246,0.67714,-0.24532
acuracia,0.92655,0.33847,-0.58808
precisao_macro,0.91428,0.56041,-0.35387
f1_macro,0.91741,0.41477,-0.50264
f1_ponderado,0.92640,0.42581,-0.50059
mcc,0.91361,0.36879,-0.54482


== Validação × teste: por classe ==


,recall (validação),recall (teste),Δ recall,precisão (validação),precisão (teste),F1 (validação),F1 (teste)
BENIGN,0.9993,0.9806,-0.0187,0.9977,0.0273,0.9985,0.0531
LDAP,0.9784,0.9926,0.0142,0.9559,0.7597,0.9670,0.8607
MSSQL,0.8884,0.3411,-0.5473,0.9351,0.9920,0.9111,0.5076
NetBIOS,0.8458,0.9987,0.1529,0.7531,0.1440,0.7968,0.2517
Syn,0.9790,0.1617,-0.8173,0.9447,0.9999,0.9615,0.2784
UDP,0.9409,0.9081,-0.0328,0.8952,0.9999,0.9175,0.9518
UDPLag,0.8254,0.3573,-0.4681,0.9183,0.0001,0.8694,0.0002


## 4. Classes do teste que o modelo não conhece (`Portmap`)
Essas classes **não entram em nenhuma métrica principal**: o modelo só pode responder uma das 7 classes que viu. Aqui se mostra **para qual classe o modelo as empurra** e **com que confiança** (a probabilidade da classe escolhida), comparada com a confiança nos acertos e nos erros das classes conhecidas. Se a confiança nelas fosse claramente menor, daria para usar um limiar de confiança como alarme de "classe desconhecida"; se for parecida, o modelo trata a classe nova como se fosse conhecida.

As linhas são lidas direto de `prep_final_teste` (só as sem sobreposição com o treino) e passam pelo mesmo `PreprocessadorRF` do modelo (a conferência do notebook 05m confirmou que ele reproduz as views).

In [9]:
# ---------- 4.1 Classes do teste que o modelo não conhece ----------
t_sec0 = time.perf_counter(); DESC, PORT = {}, pd.DataFrame()
p_pred_te = P_te[np.arange(len(pred_te)), pred_te]; acerto_te = (pred_te == y_te)
pcts = lambda a: [round(float(np.percentile(a, q_)), 3) for q_ in (10, 50, 90)]                  # percentis 10/50/90 como números comuns
if FORA:
    cols_raw = ", ".join(f"ifNull(toFloat64({q(c)}), nan) AS {q(c)}" for c in prep.features); linhas_p = []
    for classe, n_c in FORA.items():
        rot = f"({lit(classe)}, {lit('DrDoS_' + classe)})"
        dfp = ler_csv_com_espera(f"SELECT {cols_raw} FROM {TAB_TESTE} WHERE label_multi IN {rot} AND sobrepoe_treino = 0", prep.features)
        assert len(dfp) == n_c, f"{classe}: leu {len(dfp):,} linhas, esperava {n_c:,}"
        Pp = modelo.predict_proba(prep.transformar(dfp[prep.features].to_numpy(float))).astype(np.float32); pp = Pp.argmax(axis=1); pmax = Pp.max(axis=1); DESC[classe] = (Pp, pp, pmax)
        for k in range(K): linhas_p.append({"classe desconhecida": classe, "prevista como": CLASSES[k], "linhas": int((pp == k).sum()), "%": 100 * float((pp == k).mean()), "prob. média da classe": float(Pp[:, k].mean()),
                                            "confiança mediana (quando prevista)": float(np.median(pmax[pp == k])) if (pp == k).any() else np.nan})
        print(f"\n== {classe}: {n_c:,} linhas (o modelo nunca viu esta classe) — para qual classe ele as empurra ==")
        display(pd.DataFrame([l for l in linhas_p if l["classe desconhecida"] == classe]).drop(columns="classe desconhecida").set_index("prevista como").round(4))
        print(f"Confiança (probabilidade da classe escolhida), percentis 10/50/90:  {classe}: {pcts(pmax)}  | acertos das classes conhecidas: {pcts(p_pred_te[acerto_te])}  | erros das classes conhecidas: {pcts(p_pred_te[~acerto_te]) if (~acerto_te).any() else "sem erros"}")
    PORT = pd.DataFrame(linhas_p)
else: print("O teste não tem classes desconhecidas pelo modelo.")
T_SEC["4.1_desconhecidas"] = time.perf_counter() - t_sec0


== Portmap: 2,702 linhas (o modelo nunca viu esta classe) — para qual classe ele as empurra ==


,linhas,%,prob. média da classe,confiança mediana (quando prevista)
prevista como,,,,
BENIGN,233,8.6232,0.0615,0.4032
LDAP,9,0.3331,0.0158,0.3349
MSSQL,50,1.8505,0.0271,0.7329
NetBIOS,2385,88.2679,0.8550,0.9910
Syn,0,0.0000,0.0073,NaN
UDP,0,0.0000,0.0049,NaN
UDPLag,25,0.9252,0.0283,0.4454


Confiança (probabilidade da classe escolhida), percentis 10/50/90:  Portmap: [0.494, 0.991, 0.998]  | acertos das classes conhecidas: [0.391, 0.798, 0.981]  | erros das classes conhecidas: [0.373, 0.403, 0.678]


## 5. Custo computacional na inferência
Vazão em **dados reais** (todo o teste): linhas ÷ soma dos tempos de previsão de cada lote, com todos os núcleos. A latência de 1 fluxo e a vazão em lote do modelo foram medidas no notebook 05m (na validação).

In [10]:
# ---------- 5.1 Custo na inferência (dados reais) ----------
t_total = float(sum(t_pred)); vazao_real = N_TESTE / t_total if t_total > 0 else float("nan")
print(f"Modelo final: {N_TESTE:,} fluxos em {t_total:.1f} s de previsão = {vazao_real:,.0f} fluxos/s (n_jobs = {modelo.n_jobs}; {len(modelo.estimators_)} árvores)")
print(f"Tempo de relógio do teste (leitura do servidor + previsão + gravação): {fmt_tempo(TEMPO_TESTE_RELOGIO)} em {N_SESSOES_TESTE} sessão(ões)")

Modelo final: 7,216,487 fluxos em 68.9 s de previsão = 104,747 fluxos/s (n_jobs = -1; 100 árvores)
Tempo de relógio do teste (leitura do servidor + previsão + gravação): 4min 54s em 1 sessão(ões)


## 6. Comparação com o modelo padrão (notebook 06m)
Compara o modelo **ajustado** (este) com o **padrão** (06m) **no mesmo teste**: métricas gerais, recall, precisão e F1 por classe, o desempenho **ataque × BENIGN** (juntando as 6 classes de ataque numa só; é um cálculo posterior, só descritivo) e uma comparação **linha a linha** (quantas linhas só um dos modelos acerta e para onde mudaram as previsões do `Syn`). As diferenças são **descritivas**: o teste tem uma só partição e muitas linhas repetidas.

In [11]:
# ---------- 6.1 Comparação com o modelo padrão (notebook 06m) ----------
COMP = {}
if PASTA_PADRAO is None: print("(o resultado do 06m não foi encontrado: a comparação fica para depois)")
else:
    pad = json.loads((PASTA_PADRAO / "metricas_teste.json").read_text(encoding="utf-8")); assert pad["n_teste"] == N_TESTE and pad["classes"] == CLASSES, "o 06m usou outro teste ou outras classes"
    print(f"Modelo padrão (06m): {PASTA_PADRAO.name} | {N_TESTE:,} linhas nos dois")
    ger = pd.DataFrame({"padrão (06m)": {k: pad["metricas"][k] for k in ordem}, "ajustado (este)": {k: esc[k] for k in ordem}})
    ger.loc["f1_macro_sem_" + ESPECIAL] = [pad["metricas_sem_classe_especial"]["f1_macro"], f1_sem_esp]; ger["Δ (ajustado − padrão)"] = ger["ajustado (este)"] - ger["padrão (06m)"]
    print("\n== Métricas gerais no teste: padrão × ajustado =="); display(ger.round(5))
    cmp_cls = pd.DataFrame({f"{m} ({lado})": [(pad["por_classe"][c][m] if lado == "padrão" else tab_cls_te.loc[c, m]) for c in CLASSES] for m in ("recall", "precisão", "F1", "ROC-AUC um-contra-resto") for lado in ("padrão", "ajustado")}, index=CLASSES)
    for m in ("recall", "precisão", "F1", "ROC-AUC um-contra-resto"): cmp_cls["Δ " + m] = cmp_cls[f"{m} (ajustado)"] - cmp_cls[f"{m} (padrão)"]
    print("== Por classe: padrão × ajustado =="); display(cmp_cls.round(4))
    # ataque x BENIGN (calculado depois de ver os resultados; só descritivo)
    assert COD["BENIGN"] == 0
    def colapsa(cm_):
        cm_ = np.asarray(cm_, float); tn, fp, fn, tp = cm_[0, 0], cm_[0, 1:].sum(), cm_[1:, 0].sum(), cm_[1:, 1:].sum(); den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        return {"recall dos ataques": tp / (tp + fn), "falso alarme (BENIGN → ataque)": fp / (fp + tn), "MCC ataque × BENIGN": (tp * tn - fp * fn) / den if den > 0 else 0.0, "ataques previstos como BENIGN": int(fn), "Syn previsto como BENIGN (%)": 100 * cm_[COD["Syn"], 0] / cm_[COD["Syn"]].sum()}
    cm_pad = np.array([[pad["matriz_confusao"][f"real {a}"][f"prev. {b}"] for b in CLASSES] for a in CLASSES], float)
    colap = pd.DataFrame({"padrão (06m)": colapsa(cm_pad), "ajustado (este)": colapsa(cm)}); print("== Ataque × BENIGN (as 6 classes de ataque juntas; cálculo posterior, descritivo) =="); display(colap.drop(index="ataques previstos como BENIGN").round(5))
    print(f"Ataques previstos como BENIGN: padrão {int(colap.loc['ataques previstos como BENIGN', 'padrão (06m)']):,} | ajustado {int(colap.loc['ataques previstos como BENIGN', 'ajustado (este)']):,}")
    # linha a linha: as duas previsões, alinhadas pelo id da linha
    d0 = np.load(PASTA_PADRAO / "predicoes_teste.npz"); ids0 = d0["id_linha"]; o0, o1 = np.argsort(ids0, kind="stable"), np.argsort(id_te, kind="stable")
    assert np.array_equal(ids0[o0], id_te[o1]), "o 06m e este teste não têm as mesmas linhas"
    y0, p0, y1, p1 = d0["y"][o0], d0["pred"][o0], y_te[o1], pred_te[o1]; assert np.array_equal(y0, y1), "as classes reais não coincidem entre o 06m e este teste"
    ok0, ok1 = (p0 == y0), (p1 == y1); linhas = []
    for nome, msk in [("todas as classes", np.ones(len(y1), bool))] + [(c, y1 == COD[c]) for c in CLASSES]:
        a, b, c_, d_ = int((ok0 & ok1 & msk).sum()), int((~ok0 & ok1 & msk).sum()), int((ok0 & ~ok1 & msk).sum()), int((~ok0 & ~ok1 & msk).sum())
        linhas.append({"classe real": nome, "linhas": int(msk.sum()), "certas nos dois": a, "só o ajustado acerta": b, "só o padrão acerta": c_, "erram nos dois": d_, "saldo (ajustado − padrão)": b - c_})
    par = pd.DataFrame(linhas); print("== Linha a linha: quem acerta (mesmas linhas nos dois) =="); display(par)
    ms = (y1 == COD["Syn"]); tr = pd.crosstab(pd.Series(np.array(CLASSES)[p0[ms]], name="padrão previu"), pd.Series(np.array(CLASSES)[p1[ms]], name="ajustado previu"))
    print("== Linhas reais Syn: para onde mudou a previsão (linhas = padrão; colunas = ajustado) =="); display(tr)
    COMP = {"geral": ger, "classe": cmp_cls, "colapsado": colap, "pareada": par, "syn": tr}; del d0, ids0, o0, o1, y0, p0, y1, p1, ok0, ok1, ms
    g = cmp_cls["Δ recall"]; print(f"Maior ganho de recall: {g.max():+.4f} ({g.idxmax()}) | maior queda: {g.min():+.4f} ({g.idxmin()}) | F1 macro: {ger.loc['f1_macro', 'Δ (ajustado − padrão)']:+.5f} | MCC: {ger.loc['mcc', 'Δ (ajustado − padrão)']:+.5f}")


Modelo padrão (06m): rfm_20261101_210334 | 7,216,487 linhas nos dois

== Métricas gerais no teste: padrão × ajustado ==


,padrão (06m),ajustado (este),Δ (ajustado − padrão)
roc_auc_ovr_macro,0.91328,0.92483,0.01155
acuracia_balanceada,0.69079,0.67714,-0.01365
acuracia,0.39168,0.33847,-0.05320
precisao_macro,0.69269,0.56041,-0.13227
f1_macro,0.56262,0.41477,-0.14786
f1_ponderado,0.51081,0.42581,-0.08500
mcc,0.43242,0.36879,-0.06362
f1_macro_sem_UDPLag,0.65637,0.48386,-0.17251


== Por classe: padrão × ajustado ==


,recall (padrão),recall (ajustado),precisão (padrão),precisão (ajustado),F1 (padrão),F1 (ajustado),ROC-AUC um-contra-resto (padrão),ROC-AUC um-contra-resto (ajustado),Δ recall,Δ precisão,Δ F1,Δ ROC-AUC um-contra-resto
BENIGN,0.9824,0.9806,0.9312,0.0273,0.9561,0.0531,0.9933,0.9908,-0.0018,-0.9040,-0.9031,-0.0026
LDAP,0.9632,0.9926,0.7799,0.7597,0.8619,0.8607,0.9968,0.9981,0.0294,-0.0202,-0.0012,0.0014
MSSQL,0.3681,0.3411,0.9901,0.9920,0.5367,0.5076,0.9152,0.9978,-0.0270,0.0019,-0.0291,0.0826
NetBIOS,0.9990,0.9987,0.1480,0.1440,0.2577,0.2517,0.9964,0.9996,-0.0003,-0.0040,-0.0060,0.0032
Syn,0.2445,0.1617,0.9999,0.9999,0.3930,0.2784,0.9996,0.9999,-0.0829,-0.0000,-0.1146,0.0003
UDP,0.8744,0.9081,0.9996,0.9999,0.9328,0.9518,0.9954,0.9998,0.0337,0.0003,0.0190,0.0044
UDPLag,0.4039,0.3573,0.0001,0.0001,0.0001,0.0002,0.4964,0.4879,-0.0467,0.0000,0.0001,-0.0085


== Ataque × BENIGN (as 6 classes de ataque juntas; cálculo posterior, descritivo) ==


,padrão (06m),ajustado (este)
recall dos ataques,0.99943,0.72547
falso alarme (BENIGN → ataque),0.01759,0.01937
MCC ataque × BENIGN,0.95613,0.13825
Syn previsto como BENIGN (%),0.07452,40.20921


Ataques previstos como BENIGN: padrão 4,078 | ajustado 1,965,726
== Linha a linha: quem acerta (mesmas linhas nos dois) ==


,classe real,linhas,certas nos dois,só o ajustado acerta,só o padrão acerta,erram nos dois,saldo (ajustado − padrão)
0,todas as classes,7216487,2251619,190960,574905,4199003,-383945
1,BENIGN,56216,54907,220,320,769,-100
2,LDAP,112491,108133,3523,218,617,3305
3,MSSQL,854242,280264,11084,34182,528712,-23098
4,NetBIOS,91175,91041,17,44,73,-27
5,Syn,4887060,672812,117325,522255,3574668,-404930
6,UDP,1214746,1044269,58785,17854,93838,40931
7,UDPLag,557,193,6,32,326,-26


== Linhas reais Syn: para onde mudou a previsão (linhas = padrão; colunas = ajustado) ==


ajustado previu,BENIGN,LDAP,MSSQL,NetBIOS,Syn,UDP,UDPLag
padrão previu,,,,,,,
BENIGN,1537,0,0,13,2092,0,0
LDAP,68,17,2,52,0,0,0
MSSQL,21,0,744,121,0,0,23
NetBIOS,468,1,5,1855,0,0,31
Syn,148882,0,0,15,672812,0,373358
UDP,7,0,0,44,0,8,1
UDPLag,1814065,0,0,389,115233,0,1755196


Maior ganho de recall: +0.0337 (UDP) | maior queda: -0.0829 (Syn) | F1 macro: -0.14786 | MCC: -0.06362


## 7. Gravação e conferências
Grava as previsões, as métricas, a matriz de confusão, a comparação validação × teste, a análise das classes desconhecidas, o custo e os gráficos, e acrescenta uma linha a `registro_testes.csv`. Depois confere tudo de forma independente. Cada arquivo só é **criado se ainda não existir** (nunca sobrescreve). **O `metricas_teste.json` é gravado por último, só se todas as conferências passarem**: é ele que aciona a trava de uso único.

In [12]:
# ---------- 6.1 Gravação e conferências ----------
t_sec0 = time.perf_counter()
gravar("predicoes_teste.npz", lambda p_: np.savez_compressed(p_, id_linha=id_te, y=y_te, pred=pred_te, p_pred=p_pred_te, p_real=P_te[np.arange(len(y_te)), y_te], classes=np.array(CLASSES)))
gravar_csv(tab_te.reset_index(names="metrica"), "metricas_teste.csv", index=False); gravar_csv(tab_cls_te.reset_index(names="classe"), "metricas_por_classe_teste.csv", index=False)
gravar_csv(cm_te.reset_index(names="classe_real"), "matriz_confusao_teste.csv", index=False); gravar_csv(cm_lin.reset_index(names="classe_real"), "matriz_confusao_teste_fracao_da_real.csv", index=False)
gravar_csv(vs_cls.reset_index(names="classe"), "validacao_vs_teste.csv", index=False); gravar_csv(vs_ger.reset_index(names="metrica"), "validacao_vs_teste_geral.csv", index=False)
if len(PORT): gravar_csv(PORT, "portmap_teste.csv", index=False)
gravar_json({"vazao_dados_reais_fluxos_s": vazao_real, "tempo_previsao_total_s": t_total, "tempo_teste_relogio_s": TEMPO_TESTE_RELOGIO, "sessoes_teste": N_SESSOES_TESTE, "n_jobs_modelo": modelo.n_jobs, "n_estimators": len(modelo.estimators_)}, "custo_inferencia_teste.json")
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
    im = ax[0].imshow(cm_lin.values, cmap="Blues", vmin=0, vmax=1); ax[0].set_xticks(range(K)); ax[0].set_yticks(range(K)); ax[0].set_xticklabels(CLASSES, rotation=45, ha="right"); ax[0].set_yticklabels(CLASSES)
    for i in range(K):
        for j in range(K): ax[0].text(j, i, f"{cm_lin.values[i, j]:.2f}" if cm_lin.values[i, j] >= 0.005 else "", ha="center", va="center", fontsize=8, color="white" if cm_lin.values[i, j] > 0.5 else "black")
    ax[0].set_xlabel("classe prevista"); ax[0].set_ylabel("classe real"); ax[0].set_title("Matriz de confusão no teste (fração da classe real)"); fig.colorbar(im, ax=ax[0], fraction=0.046)
    x = np.arange(K); ax[1].bar(x - 0.2, vs_cls["recall (validação)"], 0.4, label="validação (mesmo dia)"); ax[1].bar(x + 0.2, vs_cls["recall (teste)"], 0.4, label="teste (outro dia)")
    ax[1].set_xticks(x); ax[1].set_xticklabels(CLASSES, rotation=45, ha="right"); ax[1].set_ylim(0, 1.05); ax[1].set_ylabel("recall"); ax[1].set_title("Recall por classe: validação × teste"); ax[1].legend(loc="lower right"); ax[1].grid(axis="y", alpha=.3)
    fig.tight_layout(); gravar("teste_multiclasse.png", lambda p_: fig.savefig(p_, dpi=130)); plt.close(fig)
except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")
T_SEC["6.1_gravacao"] = time.perf_counter() - t_sec0
gravar_csv(pd.DataFrame({"etapa": list(T_SEC), "segundos": [round(v, 1) for v in T_SEC.values()]}), "tempos_etapas_teste.csv", index=False)

if COMP:
    gravar_csv(COMP["geral"].reset_index(names="metrica"), "comparacao_ajustado_vs_padrao_geral.csv", index=False); gravar_csv(COMP["classe"].reset_index(names="classe"), "comparacao_ajustado_vs_padrao_por_classe.csv", index=False)
    gravar_csv(COMP["colapsado"].reset_index(names="metrica"), "comparacao_ajustado_vs_padrao_ataque_benign.csv", index=False); gravar_csv(COMP["pareada"], "comparacao_ajustado_vs_padrao_linha_a_linha.csv", index=False)
    gravar_csv(COMP["syn"].reset_index(), "comparacao_ajustado_vs_padrao_syn_transicoes.csv", index=False)

# ---------- conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Previsões = linhas da view de teste, com ids únicos", len(y_te) == N_TESTE == len(np.unique(id_te)), f"({len(y_te):,})")
verifica("Linhas por classe = as da tabela do notebook 03m (teste sem sobreposição)", all(int((y_te == COD[c]).sum()) == N_CLASSE[c] for c in CLASSES))
verifica("Probabilidades finitas, entre 0 e 1, e cada linha soma 1", bool(np.isfinite(P_te).all() and (P_te >= 0).all() and (P_te <= 1 + 1e-6).all() and np.abs(P_te.sum(axis=1) - 1.0).max() < 1e-4))
verifica("Matriz de confusão fecha com o total e com as linhas por classe", int(cm.sum()) == N_TESTE and all(int(cm[COD[c]].sum()) == N_CLASSE[c] for c in CLASSES))
verifica("Recall por classe = diagonal ÷ linhas da classe, calculado de forma independente", all(abs(float(((y_te == k) & (pred_te == k)).sum() / (y_te == k).sum()) - arr["recall"][k]) < 1e-12 for k in range(K)))
verifica("O IC95% do recall contém o recall de cada classe", all(tab_cls_te.loc[c, "IC95% recall (inf.)"] - 1e-9 <= tab_cls_te.loc[c, "recall"] <= tab_cls_te.loc[c, "IC95% recall (sup.)"] + 1e-9 for c in CLASSES))
d_lido = np.load(PASTA_RUN / "predicoes_teste.npz"); es2, _ = metricas_multi_grande(d_lido["y"], P_te)
verifica("predicoes_teste.npz relido reproduz as classes previstas e as métricas", bool((d_lido["pred"] == pred_te).all()) and abs(float((d_lido["pred"] == d_lido["y"]).mean()) - esc["acuracia"]) < 1e-12 and abs(es2["f1_macro"] - esc["f1_macro"]) < 1e-12)
verifica("O modelo conhece exatamente as classes 0..K-1, e nenhuma classe desconhecida entrou nas métricas principais", list(modelo.classes_) == list(range(K)) and int(y_te.max()) == K - 1 and not (set(FORA) & set(CLASSES)))
verifica("Classes desconhecidas: todas as linhas foram lidas e previstas", all(len(DESC[c][1]) == FORA[c] for c in FORA) if FORA else True, f"({FORA})" if FORA else "(nenhuma)")
verifica("Arquivos esperados gravados", all((PASTA_RUN / f).exists() for f in ("predicoes_teste.npz", "metricas_teste.csv", "metricas_por_classe_teste.csv", "matriz_confusao_teste.csv", "validacao_vs_teste.csv", "custo_inferencia_teste.json", "tempos_teste.csv")))
verifica("Custo medido: tempos positivos", t_total > 0 and TEMPO_TESTE_RELOGIO > 0)
verifica("Nenhum modelo foi treinado neste notebook (o treino está bloqueado e nenhuma tentativa foi feita)", len(TREINOS_BLOQUEADOS) == 0 and getattr(RandomForestClassifier.fit, "_bloqueado", False))
_depois = {str(f.relative_to(PASTA_RUN)): (f.stat().st_size, f.stat().st_mtime_ns) for f in PASTA_RUN.rglob("*") if f.is_file()}; _mexidos = [n for n, v in SNAP_ANTES.items() if _depois.get(n) != v]
verifica(f"Nenhum dos {len(SNAP_ANTES)} arquivos que já existiam na pasta do modelo foi alterado, sobrescrito ou apagado", not _mexidos, f"({_mexidos[:3]})" if _mexidos else "")
verifica("Comparação com o 06m: as linhas coincidem e os totais fecham" if COMP else "Comparação com o 06m: pulada (resultado do 06m não encontrado)", (int(COMP["pareada"].loc[0, "linhas"]) == N_TESTE and int((COMP["pareada"].loc[0, ["certas nos dois", "só o ajustado acerta", "só o padrão acerta", "erram nos dois"]]).sum()) == N_TESTE) if COMP else True)
assert all(ok), "há checagens reprovadas acima (o metricas_teste.json NÃO foi gravado, então o teste pode ser refeito sem novas leituras)"

gravar_json({"run_id": config["run_id"], "data_teste": datetime.now().isoformat(timespec="seconds"), "n_teste": N_TESTE, "classes": CLASSES, "metricas": esc, "metricas_sem_classe_especial": {"classe": ESPECIAL, "f1_macro": f1_sem_esp, "acuracia_balanceada": rec_sem_esp},
             "por_classe": tab_cls_te.to_dict(orient="index"), "matriz_confusao": cm_te.to_dict(orient="index"), "validacao_vs_teste": vs_ger.to_dict(orient="index"), "classes_desconhecidas": FORA,
             "custo": {"vazao_dados_reais_fluxos_s": vazao_real, "tempo_previsao_total_s": t_total, "tempo_teste_relogio_s": TEMPO_TESTE_RELOGIO}}, RESULTADO.name)
reg = {"run_id": config["run_id"], "data_teste": datetime.now().isoformat(timespec="seconds"), "n_teste": N_TESTE, "roc_auc_ovr_macro": esc["roc_auc_ovr_macro"], "acuracia_balanceada": esc["acuracia_balanceada"], "acuracia": esc["acuracia"], "f1_macro": esc["f1_macro"],
       f"f1_macro_sem_{ESPECIAL}": f1_sem_esp, "mcc": esc["mcc"], **{f"recall_{c}": float(arr["recall"][COD[c]]) for c in CLASSES}, "vazao_dados_reais_fluxos_s": vazao_real, "tempo_teste_relogio_s": TEMPO_TESTE_RELOGIO, "pasta": str(PASTA_RUN)}
caminho_reg = PASTA_BASE / "registro_testes.csv"; pd.DataFrame([reg]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_RUN}")
print(f"\n⏱ Tempo total do notebook (etapas medidas): {fmt_tempo(float(sum(T_SEC.values())))}")
print(f"\nArquivos NOVOS criados nesta execução ({len(CRIADOS)}): {', '.join(CRIADOS)}")
if MANTIDOS: print(f"Arquivos que já existiam e foram MANTIDOS ({len(MANTIDOS)}): {', '.join(MANTIDOS)}")
print("\nPróximo passo: me mande o texto desta saída (e, se quiser, os arquivos comparacao_ajustado_vs_padrao_*.csv, metricas_por_classe_teste.csv, matriz_confusao_teste.csv, portmap_teste.csv e teste_multiclasse.png).")


✔ Previsões = linhas da view de teste, com ids únicos (7,216,487)
✔ Linhas por classe = as da tabela do notebook 03m (teste sem sobreposição) 
✔ Probabilidades finitas, entre 0 e 1, e cada linha soma 1 
✔ Matriz de confusão fecha com o total e com as linhas por classe 
✔ Recall por classe = diagonal ÷ linhas da classe, calculado de forma independente 
✔ O IC95% do recall contém o recall de cada classe 
✔ predicoes_teste.npz relido reproduz as classes previstas e as métricas 
✔ O modelo conhece exatamente as classes 0..K-1, e nenhuma classe desconhecida entrou nas métricas principais 
✔ Classes desconhecidas: todas as linhas foram lidas e previstas ({'Portmap': 2702})
✔ Arquivos esperados gravados 
✔ Custo medido: tempos positivos 
✔ Nenhum modelo foi treinado neste notebook (o treino está bloqueado e nenhuma tentativa foi feita) 
✔ Nenhum dos 60 arquivos que já existiam na pasta do modelo foi alterado, sobrescrito ou apagado 
✔ Comparação com o 06m: as linhas coincidem e os totais fech